# Text Classification
## This notebook outlines the usage of NLP Feature extraction (CountVectorizer, TfidfVectorizer) in classification of text documents

### Import all the necessary libraries

In [2]:
from pprint import pprint
from time import time
import logging

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline


import time
import logging
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import CountVectorizer

from gensim.models import Word2Vec, Doc2Vec, FastText
from gensim.models.doc2vec import TaggedDocument

### Choose a few categories fro the entire 20 categories

In [24]:
# Load some categories from the training set
categories = [
    'alt.atheism',
    'talk.religion.misc',
]

In [25]:
print("Loading 20 newsgroups dataset for categories:")
print(categories)

Loading 20 newsgroups dataset for categories:
['alt.atheism', 'talk.religion.misc']


### Fetch documents for these 2 categories

In [26]:
data = fetch_20newsgroups(subset='train', categories=categories)
print(f"{len(data.filenames)} documents")
print(f"{len(data.target_names)} categories")
print()

857 documents
2 categories



### Define a pipeline combining a text feature extractor with a simple classifier

In [6]:
import pandas as pd

df = pd.DataFrame(columns=[
    "algorithm",
    "feature",
    "best_score_cv",
    "best_params",
    "time_sec"
])

df.to_csv("experiment_log.csv", index=False)

In [7]:
import os
import time
import pandas as pd

def log_grid_search_result(grid_search, algorithm_name, feature_name, start_time, file="experiment_log.csv"):
    row = {
        "algorithm": algorithm_name,
        "feature": feature_name,
        "best_score_cv": grid_search.best_score_,
        "best_params": str(grid_search.best_params_),
        "time_sec": round(time.time() - start_time, 4)
    }

    df = pd.DataFrame([row])

    if os.path.exists(file):
        df.to_csv(file, mode="a", header=False, index=False)
    else:
        df.to_csv(file, mode="w", header=True, index=False)

    print("LOGGED:", row)

In [129]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from gensim.models import Word2Vec

class Word2VecVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, vector_size=100, window=5, min_count=1, workers=1, sg=0):
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.workers = workers
        self.sg = sg
        self.model = None

    def fit(self, X, y=None):
        tokenized = [str(doc).split() for doc in X]
        self.model = Word2Vec(
            sentences=tokenized,
            vector_size=self.vector_size,
            window=self.window,
            min_count=self.min_count,
            workers=self.workers,
            sg=self.sg
        )
        return self

    def transform(self, X):
        tokenized = [str(doc).split() for doc in X]
        vectors = []

        for tokens in tokenized:
            word_vectors = [self.model.wv[word] for word in tokens if word in self.model.wv]
            if word_vectors:
                vectors.append(np.mean(word_vectors, axis=0))
            else:
                vectors.append(np.zeros(self.vector_size))

        return np.array(vectors)

In [192]:
from sklearn.base import BaseEstimator, TransformerMixin
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import numpy as np

class Doc2VecVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, vector_size=100, window=5, min_count=1, epochs=20, workers=1):
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.epochs = epochs
        self.workers = workers
        self.model = None

    def fit(self, X, y=None):
        # تبدیل متن‌ها به توکن
        tokenized = [str(doc).split() for doc in X]

        # ساخت TaggedDocument
        tagged_data = [
            TaggedDocument(words=doc, tags=[str(i)])
            for i, doc in enumerate(tokenized)
        ]

        # ساخت مدل
        self.model = Doc2Vec(
            vector_size=self.vector_size,
            window=self.window,
            min_count=self.min_count,
            workers=self.workers,
            epochs=self.epochs
        )

        # build vocab + train
        self.model.build_vocab(tagged_data)
        self.model.train(
            tagged_data,
            total_examples=self.model.corpus_count,
            epochs=self.model.epochs
        )

        return self

    def transform(self, X):
        tokenized = [str(doc).split() for doc in X]

        vectors = []
        for doc in tokenized:
            vec = self.model.infer_vector(doc)
            vectors.append(vec)

        return np.array(vectors)

In [220]:
from sklearn.base import BaseEstimator, TransformerMixin
from gensim.models import FastText
import numpy as np

class FastTextVectorizer(BaseEstimator, TransformerMixin):
    def __init__(self, vector_size=100, window=5, min_count=1, sg=0, epochs=10, workers=1):
        self.vector_size = vector_size
        self.window = window
        self.min_count = min_count
        self.sg = sg
        self.epochs = epochs
        self.workers = workers
        self.model = None

    def fit(self, X, y=None):
        # tokenize
        tokenized = [str(doc).split() for doc in X]

        # train FastText model
        self.model = FastText(
            sentences=tokenized,
            vector_size=self.vector_size,
            window=self.window,
            min_count=self.min_count,
            sg=self.sg,
            workers=self.workers,
            epochs=self.epochs
        )

        return self

    def transform(self, X):
        tokenized = [str(doc).split() for doc in X]
        vectors = []

        for tokens in tokenized:
            word_vecs = [self.model.wv[word] for word in tokens if word in self.model.wv]
            if word_vecs:
                vectors.append(np.mean(word_vecs, axis=0))
            else:
                vectors.append(np.zeros(self.vector_size))

        return np.array(vectors)

In [ ]:
algorithms = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Support Vector Machine": LinearSVC(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "SGD Classifier": SGDClassifier(tol=1e-3)
}

feature_names = [
    "CountVectorizer",
    "Word2Vec",
    "Doc2Vec",
    "FastText"
]

In [8]:

pipeline = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier(tol=1e-3)),
])

### Specify parameter grid
- 'vect__max_df': (0.5, 0.75, 1.0)
- 'vect__max_features': (None, 5000, 10000, 50000)
- 'vect__ngram_range': ((1, 1), (1, 2))
- 'tfidf__use_idf': (True, False)
- 'tfidf__norm': ('l1', 'l2')
- 'clf__max_iter': (20,)
- 'clf__alpha': (0.00001, 0.000001)
- 'clf__penalty': ('l2', 'elasticnet')
- 'clf__max_iter': (10, 50, 80)

### Find the best parameters for both the feature extraction and the classifier

In [9]:
parameters = {
    'vect__max_df': (0.5, 0.75, 1.0),
    # 'vect__max_features': (None, 5000, 10000, 50000),
    'vect__ngram_range': ((1, 1), (1, 2)),  # unigrams or bigrams
    # 'tfidf__use_idf': (True, False),
    # 'tfidf__norm': ('l1', 'l2'),
    'clf__max_iter': (20,),
    'clf__alpha': (0.00001, 0.000001),
    'clf__penalty': ('l2', 'elasticnet'),
    # 'clf__max_iter': (10, 50, 80),
}

### Build a GridSearch with the pipeline and parameter grid

In [10]:
grid_search = GridSearchCV(pipeline, parameters, cv=5,
                           n_jobs=-1, verbose=1)

### Start the grid search

In [250]:
start = time.time()

grid_search.fit(data.data, data.target)

log_grid_search_result(
    grid_search=grid_search,
    algorithm_name="SGDClassifier",
    feature_name="fasttest",
    start_time=start
)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
LOGGED: {'algorithm': 'SGDClassifier', 'feature': 'fasttest', 'best_score_cv': np.float64(0.6043383652930776), 'best_params': "{'clf__alpha': 0.001, 'clf__loss': 'hinge', 'vect__vector_size': 50}", 'time_sec': 496.791}


### Best Score

In [251]:
print("Best score: %0.3f" % grid_search.best_score_)

Best score: 0.604


### Best Parameter

In [252]:
print("Best parameters set:")
best_parameters = grid_search.best_estimator_.get_params()
for param_name in sorted(parameters.keys()):
    print("\t%s: %r" % (param_name, best_parameters[param_name]))

Best parameters set:
	clf__alpha: 0.001
	clf__loss: 'hinge'
	vect__vector_size: 50


In [253]:
import pandas as pd

df = pd.read_csv("experiment_log.csv")
print(df)

                 algorithm          feature  best_score_cv  \
0       LogisticRegression  CountVectorizer       0.931151   
1            SGDClassifier  CountVectorizer       0.950979   
2            SGDClassifier         Word2Vec       0.578750   
3       LogisticRegression         Word2Vec       0.590460   
4                LinearSVC         Word2Vec       0.621923   
5            MultinomialNB         Word2Vec       0.574153   
6   DecisionTreeClassifier         Word2Vec       0.578730   
7            MultinomialNB  CountVectorizer       0.946314   
8                LinearSVC  CountVectorizer       0.947484   
9   DecisionTreeClassifier  CountVectorizer       0.886788   
10  DecisionTreeClassifier          Doc2Vec       0.632415   
11  DecisionTreeClassifier          Doc2Vec       0.641663   
12               LinearSVC          Doc2Vec       0.696695   
13      LogisticRegression          Doc2Vec       0.702509   
14           MultinomialNB          Doc2Vec       0.562424   
15      

In [254]:
df = pd.read_csv("experiment_log.csv")
df = df.sort_values("best_score_cv", ascending=False)
print(df)

                 algorithm          feature  best_score_cv  \
1            SGDClassifier  CountVectorizer       0.950979   
8                LinearSVC  CountVectorizer       0.947484   
7            MultinomialNB  CountVectorizer       0.946314   
0       LogisticRegression  CountVectorizer       0.931151   
9   DecisionTreeClassifier  CountVectorizer       0.886788   
13      LogisticRegression          Doc2Vec       0.702509   
12               LinearSVC          Doc2Vec       0.696695   
17               LinearSVC         fasttest       0.694308   
16      LogisticRegression         fasttest       0.681423   
11  DecisionTreeClassifier          Doc2Vec       0.641663   
10  DecisionTreeClassifier          Doc2Vec       0.632415   
4                LinearSVC         Word2Vec       0.621923   
18  DecisionTreeClassifier         fasttest       0.617265   
15           MultinomialNB         fasttest       0.609051   
19           SGDClassifier         fasttest       0.604338   
3       

### Choose the best model

In [255]:
best = df.iloc[0]
print("Best Model:")
print(best)

Best Model:
algorithm                 SGDClassifier
feature                 CountVectorizer
best_score_cv                  0.950979
best_params      {'vect__max_df': 0.75}
time_sec                         0.9527
Name: 1, dtype: object


### Use the model to classify a piece of text

In [42]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import SGDClassifier

best_pipeline = Pipeline([
    ('vect', CountVectorizer(max_df=0.75)),
    ('tfidf', TfidfTransformer()),
    ('clf', SGDClassifier(tol=1e-3, random_state=21))
])

best_pipeline.fit(data.data, data.target)
sample_texts = [
    "The graphics are amazing and the gameplay is very smooth.",
    "My computer keeps crashing and this program is terrible.",
    "Religion is a myth created by humans, and belief in God has no logical or scientific basis.",
    "There is no scientific evidence for the existence of God, and religion is just a human invention.",
    "God does not exist and religion is completely false and irrational.",
    "Different religions have various beliefs about God and spirituality, and people interpret them in many ways."
]

predictions = best_pipeline.predict(sample_texts)

for text, pred in zip(sample_texts, predictions):
    print("Text:", text)
    print("Predicted class:", data.target_names[pred])
    print("-" * 50)

Text: The graphics are amazing and the gameplay is very smooth.
Predicted class: talk.religion.misc
--------------------------------------------------
Text: My computer keeps crashing and this program is terrible.
Predicted class: talk.religion.misc
--------------------------------------------------
Text: Religion is a myth created by humans, and belief in God has no logical or scientific basis.
Predicted class: alt.atheism
--------------------------------------------------
Text: There is no scientific evidence for the existence of God, and religion is just a human invention.
Predicted class: talk.religion.misc
--------------------------------------------------
Text: God does not exist and religion is completely false and irrational.
Predicted class: alt.atheism
--------------------------------------------------
Text: Different religions have various beliefs about God and spirituality, and people interpret them in many ways.
Predicted class: talk.religion.misc
-------------------------